# Snæfellsjökull glacier outlines, ~1890 to present

This notebook downloads every mapped outline of **Snæfellsjökull** (western Iceland) from the **Icelandic Meteorological Office (Veðurstofa Íslands)** and produces:

- **one outline file per year** (GeoJSON, in a subfolder)
- a combined GeoPackage of all years and a CSV table of areas
- a map of all outlines and a chart of area over time
- an **animated GIF** of the ice cap shrinking, outline by outline, with every frame also saved as a PNG

It does not need Earth Engine or any login, only internet access to the Met Office's public map server.

## Data source

The outlines come from the Icelandic national glacier inventory, which the Met Office maintains with the University of Iceland and other partners. The inventory up to 2019 is described in:

> Hannesdóttir, H., Sigurðsson, O., Þrastarson, R. H., Guðmundsson, S., Belart, J. M. C., Pálsson, F., Magnússon, E., Víkingsson, S., Kaldal, I. and Jóhannesson, T. (2020). A national glacier inventory and variations in glacier extent in Iceland from the Little Ice Age maximum to 2019. *Jökull*, 70, 1–34.

The Met Office has since added Sentinel-2 outlines for 2021, 2023 and 2025.

| | |
|---|---|
| **Server** | `https://geo.vedur.is/geoserver/wfs` (an OGC Web Feature Service, WFS) |
| **Layers** | `glaciology:glacierpolygon_<period>`, one layer per mapping period, covering all of Iceland's glaciers |
| **Glacier ID** | `G336220E64811N`, the GLIMS ID for Snæfellsjökull |
| **License** | [Creative Commons Attribution 4.0 (CC-BY)](https://creativecommons.org/licenses/by/4.0/). Credit *Veðurstofa Íslands (Icelandic Meteorological Office)* when you use or share the outlines. |
| **Browse online** | [Jöklavefsjá glacier map](https://islenskirjoklar.is/index.html#/page/map) |

These outlines are **mapped by glaciologists** from aerial photos, lidar, elevation models, satellite images and (for ~1890) moraines. They are more reliable than the automatic snow and ice classification in `../Snæfellsjökull_imagery/`, which also counts leftover seasonal snow.

The notebook finds the available layers when it runs, so it picks up new periods (e.g. a 2027 outline) without code changes.

## Outputs

Everything is written to `outputs/` next to this notebook.

| Path | Contents |
|---|---|
| `outputs/outlines/snaefellsjokull_<year>.geojson` | One file per mapped year: the ice cap as a single (multi)polygon, in WGS84 latitude/longitude (EPSG:4326), with the attributes below |
| `outputs/snaefellsjokull_outlines_all_years.gpkg` | All years in one layer, in the Icelandic grid ISN93 (EPSG:3057), for GIS work |
| `outputs/snaefellsjokull_outline_areas.csv` | One row per year: date, source, area, change |
| `outputs/snaefellsjokull_outlines_map.png` | Map of all outlines |
| `outputs/snaefellsjokull_area_chart.png` | Area over time |
| `outputs/snaefellsjokull_outlines.gif` | The animation |
| `outputs/snaefellsjokull_outlines_frames/1.png, 2.png, …` | Every GIF frame as a PNG, in year order |
| `outputs/README.md` | Short description of the files, with attribution |

**Attributes in each outline file**

| Field | Meaning |
|---|---|
| `year` | Year of the outline, as a number. Used for sorting and file names. |
| `label` | Year as published, e.g. `~1890` for the Little Ice Age maximum |
| `acq_date` | Date of the source image or survey, where known. For ~1890 this is a placeholder (`1890-01-01`). |
| `source` | What the outline was mapped from (aerial images, lidar, Sentinel-2, moraines, …) |
| `analyst` | Who mapped it |
| `layer` | The WFS layer it came from |
| `n_polygons` | How many separate pieces were merged into this outline (see notes below) |
| `area_km2` | Area in km², calculated in ISN93 (EPSG:3057) |
| `change_km2` | Change since the previous outline |
| `pct_of_first` | Area as a percentage of the earliest outline (~1890) |

### Figure style

All figures use one minimal black-and-white style: white background, black text and lines, greys for secondary elements, no gridlines and no top or right axis borders. Satellite images keep their natural colours, because the colour is the data.

## 1. Setup

In [ ]:
# Install dependencies (run once)
%pip install -q geopandas requests matplotlib numpy pandas pillow

In [ ]:
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Image as ShowImage
from PIL import Image

In [ ]:
from cycler import cycler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.prop_cycle": cycler(color=["black"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.8,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "font.family": "sans-serif",
    "font.size": 10,
    "lines.linewidth": 1.2,
    "legend.frameon": False,
})

GREY = "#9a9a9a"
LIGHT_GREY = "#d4d4d4"

## 2. Settings

To run this for **another Icelandic glacier**, change `GLACIER_ID`, `GLACIER_NAME` and `SLUG`. GLIMS IDs are shown in the [Jöklavefsjá glacier map](https://islenskirjoklar.is/index.html#/page/map) and in Table 1 of Hannesdóttir et al. (2020).

In [ ]:
WFS_URL = "https://geo.vedur.is/geoserver/wfs"

GLACIER_ID = "G336220E64811N"
GLACIER_NAME = "Snæfellsjökull"
SLUG = "snaefellsjokull"  # used in file names

OUT_DIR = Path("outputs")
OUTLINE_DIR = OUT_DIR / "outlines"

MAP_CRS = "EPSG:3057"     # ISN93 / Lambert 1993, the Icelandic national grid; used for areas and maps
OUTPUT_CRS = "EPSG:4326"  # WGS84 latitude/longitude for the per-year GeoJSON files

GIF_FRAME_MS = 1200       # how long each outline stays on screen in the GIF
GIF_LAST_FRAME_MS = 3500  # hold the final frame longer before the animation loops

ATTRIBUTION = "Outlines: Veðurstofa Íslands (Icelandic Met Office), CC-BY 4.0"

OUTLINE_DIR.mkdir(parents=True, exist_ok=True)

## 3. Find the available outline layers

The server's capabilities document lists every layer it offers. We keep the polygon layers (`glacierpolygon_*`). The server also has matching `glacieroutline_*` layers, but those are lines, which can't be used for areas.

In [ ]:
def list_polygon_layers():
    """Return {layer name: layer title} for every glacier polygon layer on the server."""
    r = requests.get(
        WFS_URL,
        params={"service": "WFS", "version": "2.0.0", "request": "GetCapabilities"},
        timeout=120,
    )
    r.raise_for_status()
    layers = {}
    for block in re.findall(r"<FeatureType[^>]*>(.*?)</FeatureType>", r.text, re.S):
        name = re.search(r"<Name>(.*?)</Name>", block)
        title = re.search(r"<Title>(.*?)</Title>", block)
        if name and name.group(1).startswith("glaciology:glacierpolygon_"):
            layers[name.group(1)] = title.group(1) if title else ""
    return layers


LAYERS = list_polygon_layers()
for name, title in LAYERS.items():
    print(f"{name:45s} {title}")

## 4. Download the Snæfellsjökull outline from each layer

Each layer covers all of Iceland, so the request uses a filter (`CQL_FILTER`) to ask the server for Snæfellsjökull only. Outlines are requested directly in ISN93, which avoids any confusion over latitude/longitude axis order.

Some layers return **more than one polygon** for the glacier. These are small separate ice patches next to the main ice cap that share its glacier ID. They are merged into one outline per year, and `n_polygons` records how many there were. The year and source come from the largest piece.

In [ ]:
def fetch_layer(layer):
    """Download this glacier's polygons from one WFS layer as a GeoDataFrame in MAP_CRS."""
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeNames": layer,
        "outputFormat": "application/json",
        "srsName": MAP_CRS,
        "CQL_FILTER": f"id='{GLACIER_ID}'",
    }
    r = requests.get(WFS_URL, params=params, timeout=120)
    r.raise_for_status()
    features = r.json()["features"]
    if not features:
        return None
    return gpd.GeoDataFrame.from_features(features, crs=MAP_CRS)


def analyst_name(props):
    # Field names differ between layers
    given = props.get("anlst_givn") or props.get("given_name") or ""
    surname = props.get("anlst_surn") or props.get("surname_of") or ""
    return f"{given} {surname}".strip()


def summarize(layer, gdf):
    """Merge one layer's pieces into a single outline record."""
    gdf = gdf.copy()
    gdf["piece_km2"] = gdf.geometry.area / 1e6
    main = gdf.loc[gdf["piece_km2"].idxmax()]

    label = str(main.get("timestr") or main.get("acq_date") or "")
    year = int(re.search(r"\d{4}", label).group())
    return {
        "year": year,
        "label": label,
        "acq_date": str(main.get("acq_date") or "").rstrip("Z"),
        "source": main.get("inst_name") or "",
        "analyst": analyst_name(main),
        "layer": layer,
        "n_polygons": len(gdf),
        "geometry": gdf.dissolve().geometry.iloc[0],
    }


records = []
for layer in LAYERS:
    gdf = fetch_layer(layer)
    if gdf is None:
        print(f"{layer}: no outline for {GLACIER_ID}")
        continue
    records.append(summarize(layer, gdf))
    print(f"{layer}: {records[-1]['label']} ({len(gdf)} polygon(s))")

outlines = gpd.GeoDataFrame(records, crs=MAP_CRS).sort_values("year").reset_index(drop=True)
outlines["glacier"] = GLACIER_NAME
outlines["glims_id"] = GLACIER_ID
outlines["area_km2"] = (outlines.geometry.area / 1e6).round(3)
outlines["change_km2"] = outlines["area_km2"].diff().round(3)
outlines["pct_of_first"] = (100 * outlines["area_km2"] / outlines["area_km2"].iloc[0]).round(1)

assert outlines["year"].is_unique, "Two layers share a year; file names would collide"
outlines.drop(columns="geometry")

## 5. Save one outline per year, plus combined files

Each year is written to its own GeoJSON file in `outputs/outlines/`. GeoJSON is read by QGIS, ArcGIS, geopandas, Google Earth Engine, Mapbox and most web maps.

In [ ]:
for _, row in outlines.iterrows():
    path = OUTLINE_DIR / f"{SLUG}_{row['year']}.geojson"
    outlines.loc[[row.name]].to_crs(OUTPUT_CRS).to_file(path, driver="GeoJSON")
    print("wrote", path)

all_years_path = OUT_DIR / f"{SLUG}_outlines_all_years.gpkg"
outlines.to_file(all_years_path, layer="outlines", driver="GPKG")

areas_path = OUT_DIR / f"{SLUG}_outline_areas.csv"
outlines.drop(columns="geometry").to_csv(areas_path, index=False)

print("wrote", all_years_path)
print("wrote", areas_path)

## 6. Map of all outlines and area over time

Maps are drawn in ISN93, so distances are true and the 1 km scale bar is accurate. Older outlines are light grey and the newest is black.

In [ ]:
# Map extent: all outlines plus a margin, made square so every frame lines up
xmin, ymin, xmax, ymax = outlines.total_bounds
cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
half = max(xmax - xmin, ymax - ymin) / 2 * 1.08
EXTENT = (cx - half, cx + half, cy - half, cy + half)  # left, right, bottom, top

# Older outlines light grey, newer outlines black
year_greys = plt.cm.Greys(np.linspace(0.3, 1.0, len(outlines)))


def setup_map(ax):
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_aspect("equal")
    ax.axis("off")


def add_scale_bar(ax, length_m=1000):
    width, height = EXTENT[1] - EXTENT[0], EXTENT[3] - EXTENT[2]
    x0, y0 = EXTENT[1] - 0.04 * width - length_m, EXTENT[2] + 0.03 * height
    ax.plot([x0, x0 + length_m], [y0, y0], color="black", linewidth=2, solid_capstyle="butt")
    ax.text(x0 + length_m / 2, y0 + 0.012 * height, f"{length_m / 1000:g} km", ha="center", va="bottom", fontsize=8)


def plot_outline(ax, geometry, **kwargs):
    gpd.GeoSeries([geometry], crs=MAP_CRS).plot(ax=ax, **kwargs)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
setup_map(ax)
for i, row in outlines.iterrows():
    gpd.GeoSeries([row.geometry], crs=MAP_CRS).boundary.plot(
        ax=ax, color=year_greys[i], linewidth=1.1, label=f"{row['label']}   {row['area_km2']:.1f} km²"
    )
add_scale_bar(ax)
ax.legend(loc="upper left", fontsize=8, title="Outline   area", title_fontsize=8, alignment="left")
ax.set_title(f"{GLACIER_NAME}: mapped outlines, {outlines['label'].iloc[0]} to {outlines['label'].iloc[-1]}")
fig.text(0.02, 0.02, ATTRIBUTION, fontsize=7, color=GREY)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{SLUG}_outlines_map.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(outlines["year"], outlines["area_km2"], "o-", color="black", markersize=4)
for _, row in outlines.iterrows():
    ax.annotate(f"{row['area_km2']:.1f}", (row["year"], row["area_km2"]), textcoords="offset points",
                xytext=(0, 7), ha="center", fontsize=8)
ax.set_xlabel("Year")
ax.set_ylabel("Area (km²)")
ax.set_ylim(0, outlines["area_km2"].max() * 1.15)
ax.set_title(f"{GLACIER_NAME} area from mapped outlines")
fig.text(0.99, 0.01, ATTRIBUTION, ha="right", fontsize=7, color=GREY)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{SLUG}_area_chart.png", dpi=150)
plt.show()

## 7. Animated GIF

Each frame shows one outline: the current year in black with a light grey fill, earlier years as thin grey lines, and the year marked on the area chart to the right. Every frame is also saved as `outputs/snaefellsjokull_outlines_frames/<n>.png`, numbered from 1 in year order.

The mapped years are **unevenly spaced** (a 55-year gap from ~1890 to 1945, then 2 years apart since 2017). Every frame is held for the same time, so read the timing from the year label and the chart rather than from the animation's speed.

In [ ]:
def figure_to_image(fig):
    """Render a matplotlib figure to a PIL image and close it."""
    fig.canvas.draw()
    image = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())).convert("RGB")
    plt.close(fig)
    return image


def save_gif(frames, gif_path, durations_ms):
    """Write an animated GIF, and every frame as a PNG in <gif name>_frames/ (1.png, 2.png, ...)."""
    frames_dir = gif_path.with_name(gif_path.stem + "_frames")
    frames_dir.mkdir(parents=True, exist_ok=True)
    for old in frames_dir.glob("*.png"):
        old.unlink()
    for i, frame in enumerate(frames, start=1):
        frame.save(frames_dir / f"{i}.png")

    palette_frames = [f.convert("P", palette=Image.ADAPTIVE, colors=256) for f in frames]
    palette_frames[0].save(gif_path, save_all=True, append_images=palette_frames[1:],
                           duration=durations_ms, loop=0, optimize=True)
    print(f"wrote {gif_path} ({len(frames)} frames, {gif_path.stat().st_size / 1e6:.1f} MB)")
    print(f"wrote {len(frames)} frames to {frames_dir}/")

In [ ]:
def outline_frame(i):
    """Draw frame i (one outline year) and return it as a PIL image."""
    row = outlines.iloc[i]
    fig = plt.figure(figsize=(11, 6), dpi=100)
    grid = fig.add_gridspec(1, 2, width_ratios=[1.2, 1], wspace=0.2, left=0.03, right=0.97, top=0.9, bottom=0.12)
    ax_map = fig.add_subplot(grid[0])
    ax_chart = fig.add_subplot(grid[1])

    # Map
    setup_map(ax_map)
    for j in range(i):
        gpd.GeoSeries([outlines.geometry.iloc[j]], crs=MAP_CRS).boundary.plot(
            ax=ax_map, color=LIGHT_GREY, linewidth=0.8
        )
    plot_outline(ax_map, row.geometry, facecolor="#efefef", edgecolor="black", linewidth=1.6)
    add_scale_bar(ax_map)
    ax_map.text(0.0, 1.0, row["label"], transform=ax_map.transAxes, fontsize=26, fontweight="bold", va="top")
    ax_map.text(0.0, 0.89, f"{row['area_km2']:.1f} km²  ·  {row['pct_of_first']:.0f}% of {outlines['label'].iloc[0]}",
                transform=ax_map.transAxes, fontsize=11, va="top")
    ax_map.text(0.0, 0.84, f"Source: {row['source']}", transform=ax_map.transAxes, fontsize=8,
                color=GREY, va="top")

    # Area chart with the current year highlighted
    ax_chart.plot(outlines["year"], outlines["area_km2"], "-", color=LIGHT_GREY, zorder=1)
    ax_chart.plot(outlines["year"][: i + 1], outlines["area_km2"][: i + 1], "o-", color="black",
                  markersize=4, zorder=2)
    ax_chart.plot([row["year"]], [row["area_km2"]], "o", markersize=11, markerfacecolor="white",
                  markeredgecolor="black", markeredgewidth=1.5, zorder=3)
    ax_chart.set_ylim(0, outlines["area_km2"].max() * 1.12)
    ax_chart.set_xlim(outlines["year"].min() - 5, outlines["year"].max() + 5)
    ax_chart.set_ylabel("Area (km²)")
    ax_chart.set_title(f"{GLACIER_NAME} area")

    fig.text(0.03, 0.03, ATTRIBUTION, fontsize=7, color=GREY)
    return figure_to_image(fig)


frames = [outline_frame(i) for i in range(len(outlines))]
durations = [GIF_FRAME_MS] * (len(frames) - 1) + [GIF_LAST_FRAME_MS]
gif_path = OUT_DIR / f"{SLUG}_outlines.gif"
save_gif(frames, gif_path, durations)
ShowImage(filename=str(gif_path))

## 8. README for the output folder

In [ ]:
table = "\n".join(
    f"| {r['label']} | {r['acq_date']} | {r['source']} | {r['area_km2']:.2f} | `outlines/{SLUG}_{r['year']}.geojson` |"
    for _, r in outlines.iterrows()
)

readme = f"""# {GLACIER_NAME} glacier outlines

Mapped outlines of {GLACIER_NAME} (GLIMS ID {GLACIER_ID}), downloaded from the Icelandic Meteorological Office
web feature service ({WFS_URL}) by `download_outlines.ipynb`.

**License and attribution:** the outlines are published under CC-BY 4.0. Credit:
*Veðurstofa Íslands (Icelandic Meteorological Office)*. For outlines up to 2019, also cite Hannesdóttir et al. (2020),
A national glacier inventory and variations in glacier extent in Iceland from the Little Ice Age maximum to 2019,
*Jökull* 70, 1–34.

| Year | Source date | Source | Area (km²) | File |
|---|---|---|---|---|
{table}

## Files

- `outlines/`: one GeoJSON per year, WGS84 (EPSG:4326)
- `{all_years_path.name}`: all years in one layer, ISN93 (EPSG:3057)
- `{areas_path.name}`: area table
- `{SLUG}_outlines_map.png`, `{SLUG}_area_chart.png`: figures
- `{gif_path.name}`: animation; `{gif_path.stem}_frames/`: its frames as PNGs (1.png = earliest year)

Areas are calculated in ISN93 and can differ from published values in the second decimal place.
The ~1890 outline is reconstructed from moraines (Little Ice Age maximum), so it is less precise than later outlines.
"""

(OUT_DIR / "README.md").write_text(readme, encoding="utf-8")
print(readme)

## Notes and caveats

- **Different methods per year.** The outlines were mapped from very different sources: moraines (~1890), aerial photographs (1945, 1979, 1999, 2017), airborne lidar (2008), the ArcticDEM elevation model (2014) and Sentinel-2 satellite images (2019 onward). Small changes between years can be within the mapping uncertainty, especially between methods.
- **~1890 is a reconstruction.** It marks the Little Ice Age maximum, traced from terminal and lateral moraines (Evans et al.). It is not an observation from a single year.
- **Merged pieces.** Where a layer has several polygons for the glacier (`n_polygons > 1`), they are small ice patches beside the main ice cap. In the 1998–2004 layer, one small patch is dated 2002 while the main outline is 1999; the file uses 1999.
- **Character encoding.** Some text attributes in the 2014 layer come from the server with broken Icelandic letters (e.g. `Sigur�sson` for Sigurðsson). Glacier names are set from `GLACIER_NAME`, but `analyst` and `source` are left as received.
- **Areas.** Areas are calculated in ISN93 and match the published inventory to about ±0.1 km².
- **New outlines.** The Met Office has been adding a Sentinel-2 outline every two years. Rerun the notebook to pick up new layers automatically.